In [1]:
import pandas as pd
import numpy as np

RAW = "../data/raw/"

# Load only the tables needed for cleaning decisions today
orders   = pd.read_csv(RAW + "olist_orders_dataset.csv")
items    = pd.read_csv(RAW + "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW + "olist_order_payments_dataset.csv")
reviews  = pd.read_csv(RAW + "olist_order_reviews_dataset.csv")
customers = pd.read_csv(RAW + "olist_customers_dataset.csv")
products  = pd.read_csv(RAW + "olist_products_dataset.csv")
translation = pd.read_csv(RAW + "product_category_name_translation.csv")

print(f"Orders loaded: {len(orders):,} rows")

Orders loaded: 99,441 rows


In [2]:
# All date columns in orders table are currently strings — convert them
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Also convert date columns in reviews
reviews['review_creation_date']    = pd.to_datetime(reviews['review_creation_date'],    errors='coerce')
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce')

# Also convert shipping_limit_date in items
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'], errors='coerce')

# Verify conversion worked
print("=== DATE COLUMN DTYPES AFTER CONVERSION ===")
print(orders[date_cols].dtypes)
print("\nSample purchase dates:")
print(orders['order_purchase_timestamp'].head(3))

=== DATE COLUMN DTYPES AFTER CONVERSION ===
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Sample purchase dates:
0   2017-10-02 10:56:33
1   2018-07-24 20:41:37
2   2018-08-08 08:38:49
Name: order_purchase_timestamp, dtype: datetime64[ns]


In [3]:
# Delivered orders only — check date logic makes sense
delivered = orders[orders['order_status'] == 'delivered'].copy()

print(f"Delivered orders: {len(delivered):,}")

# Check: purchase date should be BEFORE delivered date
bad_dates = delivered[
    delivered['order_delivered_customer_date'] < delivered['order_purchase_timestamp']
]
print(f"Orders where delivery is BEFORE purchase (impossible): {len(bad_dates)}")

# Check: date range of the dataset
print(f"\nEarliest purchase: {orders['order_purchase_timestamp'].min()}")
print(f"Latest purchase:   {orders['order_purchase_timestamp'].max()}")
print(f"\nEarliest delivery: {delivered['order_delivered_customer_date'].min()}")
print(f"Latest delivery:   {delivered['order_delivered_customer_date'].max()}")

Delivered orders: 96,478
Orders where delivery is BEFORE purchase (impossible): 0

Earliest purchase: 2016-09-04 21:15:19
Latest purchase:   2018-10-17 17:30:18

Earliest delivery: 2016-10-11 13:46:32
Latest delivery:   2018-10-17 13:22:46


In [4]:
# CLEANING RULE 1:
# Keep only 'delivered' orders for delivery time and review analysis.
# Reason: delivery_days and is_late_delivery cannot be calculated
# reliably for orders that were not delivered.

print("=== ORDER STATUS BEFORE FILTER ===")
print(orders['order_status'].value_counts())

orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

print(f"\nRows BEFORE filter: {len(orders):,}")
print(f"Rows AFTER filter:  {len(orders_delivered):,}")
print(f"Rows removed:       {len(orders) - len(orders_delivered):,}")
print(f"Reason: Non-delivered orders cannot be used for delivery analysis")

=== ORDER STATUS BEFORE FILTER ===
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Rows BEFORE filter: 99,441
Rows AFTER filter:  96,478
Rows removed:       2,963
Reason: Non-delivered orders cannot be used for delivery analysis


In [5]:
# CLEANING RULE 2:
# Remove rows where order_delivered_customer_date is null within delivered orders.
# Reason: These are data quality issues — an order marked 'delivered'
# but with no delivered date cannot contribute to delivery_days calculation.

nulls_before = orders_delivered['order_delivered_customer_date'].isnull().sum()
print(f"Nulls in delivered_customer_date (within delivered orders): {nulls_before}")

orders_delivered = orders_delivered.dropna(subset=['order_delivered_customer_date'])

print(f"Rows after removing null delivered dates: {len(orders_delivered):,}")

Nulls in delivered_customer_date (within delivered orders): 8
Rows after removing null delivered dates: 96,470


In [6]:
# CLEANING RULE 3:
# Check for zero or negative prices and freight values

print("=== PRICE CHECKS ===")
print(f"Zero price rows:     {(items['price'] == 0).sum()}")
print(f"Negative price rows: {(items['price'] < 0).sum()}")
print(f"Min price:           {items['price'].min()}")
print(f"Max price:           {items['price'].max()}")

print("\n=== FREIGHT VALUE CHECKS ===")
print(f"Zero freight rows:     {(items['freight_value'] == 0).sum()}")
print(f"Negative freight rows: {(items['freight_value'] < 0).sum()}")
print(f"Min freight:           {items['freight_value'].min()}")
print(f"Max freight:           {items['freight_value'].max()}")

# Look at rows with zero freight
print("\n--- Sample rows with zero freight ---")
print(items[items['freight_value'] == 0].head(5))

=== PRICE CHECKS ===
Zero price rows:     0
Negative price rows: 0
Min price:           0.85
Max price:           6735.0

=== FREIGHT VALUE CHECKS ===
Zero freight rows:     383
Negative freight rows: 0
Min freight:           0.0
Max freight:           409.68

--- Sample rows with zero freight ---
                             order_id  order_item_id  \
114  00404fa7a687c8c44ca69d42695aae73              1   
258  00a870c6c06346e85335524935c600c0              1   
483  011c899816ea29773525bd3322dbb6aa              1   
508  012b3f6ab7776a8ab3443a4ad7bef2e6              1   
509  012b3f6ab7776a8ab3443a4ad7bef2e6              2   

                           product_id                         seller_id  \
114  53b36df67ebb7c41585e8d54d6772e08  7d13fca15225358621be4086e1eb0964   
258  aca2eb7d00ea1a7b8ebd4e68314663af  955fee9216a65b617aa5c0531780ce60   
483  53b36df67ebb7c41585e8d54d6772e08  7d13fca15225358621be4086e1eb0964   
508  422879e10f46682990de24d770e7f83d  1f50f920176fa81dab994f902

In [7]:
print("=== PAYMENT VALUE CHECKS ===")
print(f"Zero payment value rows:     {(payments['payment_value'] == 0).sum()}")
print(f"Negative payment value rows: {(payments['payment_value'] < 0).sum()}")
print(f"Zero installments rows:      {(payments['payment_installments'] == 0).sum()}")

print("\n--- Sample rows with zero payment value ---")
print(payments[payments['payment_value'] == 0].head(5))

print("\n--- Payment types ---")
print(payments['payment_type'].value_counts())

=== PAYMENT VALUE CHECKS ===
Zero payment value rows:     9
Negative payment value rows: 0
Zero installments rows:      2

--- Sample rows with zero payment value ---
                               order_id  payment_sequential payment_type  \
19922  8bcbe01d44d147f901cd3192671144db                   4      voucher   
36822  fa65dad1b0e818e3ccc5cb0e39231352                  14      voucher   
43744  6ccb433e00daae1283ccc956189c82ae                   4      voucher   
51280  4637ca194b6387e2d538dc89b124b0ee                   1  not_defined   
57411  00b1cb0320190ca0daa2c88b35206009                   1  not_defined   

       payment_installments  payment_value  
19922                     1            0.0  
36822                     1            0.0  
43744                     1            0.0  
51280                     1            0.0  
57411                     1            0.0  

--- Payment types ---
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_c

In [8]:
# CLEANING RULE 4:
# Some orders have more than 1 review (max was 3 from Day 3).
# Keep the LATEST review per order based on review_answer_timestamp.
# Reason: The most recent review best reflects final customer sentiment.

print(f"Total review rows before dedup: {len(reviews):,}")
print(f"Unique orders in reviews:        {reviews['order_id'].nunique():,}")

# Sort by answer timestamp descending, keep first (latest) per order
reviews_clean = (reviews
    .sort_values('review_answer_timestamp', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')
    .copy()
)

print(f"Total review rows after dedup:  {len(reviews_clean):,}")
print(f"Duplicate reviews removed:       {len(reviews) - len(reviews_clean):,}")

Total review rows before dedup: 99,224
Unique orders in reviews:        98,673
Total review rows after dedup:  98,673
Duplicate reviews removed:       551


In [9]:
# CLEANING RULE 5:
# 610 products have no category name.
# Fill with 'unknown' — do not drop, because these products still
# have valid price and freight data useful for revenue analysis.

print(f"Missing category names before: {products['product_category_name'].isnull().sum()}")

products['product_category_name'] = products['product_category_name'].fillna('unknown')

print(f"Missing category names after:  {products['product_category_name'].isnull().sum()}")
print("Filled with: 'unknown'")

Missing category names before: 610
Missing category names after:  0
Filled with: 'unknown'


In [10]:
print("""
=== CLEANING RULES SUMMARY ===

Rule 1 — Order Status Filter
  Action:  Keep only orders with status = 'delivered'
  Removed: ~2,963 non-delivered orders
  Reason:  delivery_days and is_late_delivery require actual delivery dates

Rule 2 — Null Delivered Date
  Action:  Remove delivered orders with null order_delivered_customer_date
  Removed: Any remaining nulls within delivered orders
  Reason:  Cannot calculate delivery time without an actual delivery date

Rule 3 — Zero/Negative Price & Freight
  Action:  Flag for review — keep in dataset but document
  Reason:  Some free items exist legitimately; investigate before removing

Rule 4 — Duplicate Reviews
  Action:  Keep latest review per order_id by review_answer_timestamp
  Removed: ~551 duplicate review rows
  Reason:  Most recent review best represents customer's final opinion

Rule 5 — Missing Product Categories
  Action:  Fill null category names with 'unknown'
  Removed: 0 rows
  Reason:  Products still have valid price/freight data for revenue analysis

Next step (Day 5): Join tables safely with these cleaned inputs.
""")


=== CLEANING RULES SUMMARY ===

Rule 1 — Order Status Filter
  Action:  Keep only orders with status = 'delivered'
  Removed: ~2,963 non-delivered orders
  Reason:  delivery_days and is_late_delivery require actual delivery dates

Rule 2 — Null Delivered Date
  Action:  Remove delivered orders with null order_delivered_customer_date
  Removed: Any remaining nulls within delivered orders
  Reason:  Cannot calculate delivery time without an actual delivery date

Rule 3 — Zero/Negative Price & Freight
  Action:  Flag for review — keep in dataset but document
  Reason:  Some free items exist legitimately; investigate before removing

Rule 4 — Duplicate Reviews
  Action:  Keep latest review per order_id by review_answer_timestamp
  Removed: ~551 duplicate review rows
  Reason:  Most recent review best represents customer's final opinion

Rule 5 — Missing Product Categories
  Action:  Fill null category names with 'unknown'
  Removed: 0 rows
  Reason:  Products still have valid price/freigh